# Inter-city evaluation of the NYC MultiAttrGAT

Load the pretrained NYC checkpoint and evaluate it on another city:

1. Pick a city by name (e.g. `"Cairo, Egypt"`) and derive its bounding box with OSMnx.
2. Pull the road graph from the Mapedia DB via `DBHandler.get_graph(...)`.
3. Merge `road_attributes` and build the line graph.
4. Apply the **frozen vocab + scalers** from the checkpoint (no refitting).
5. Run the same fixed-mask evaluation as `gnn.ipynb` and print the same metrics:
   `hwy_macro_f1`, `lan_macro_f1`, `onw_auroc`, `wid_mae_m`, `max_mae`, `min_mae`.

In [2]:
# =========================
# Setup: paths, imports, device
# =========================
import sys
from pathlib import Path

# notebooks/ is inside city_learning/; go up one level for `src.*` imports
CITY_LEARNING_DIR = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
REPO_ROOT = CITY_LEARNING_DIR.parents[1]

sys.path.insert(0, str(CITY_LEARNING_DIR))  # for `from src...`
sys.path.insert(0, str(REPO_ROOT))          # for `from modules import DBHandler`

import numpy as np
import pandas as pd
import torch
import osmnx as ox

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

CKPT_PATH = CITY_LEARNING_DIR / "notebooks" / "checkpoints" / "nyc_gat_multitask.pt"
print("Checkpoint:", CKPT_PATH, "exists=", CKPT_PATH.exists())

Device: cuda
Checkpoint: C:\Users\Ahmed sameh\Downloads\mapedia-master\modules\city_learning\notebooks\checkpoints\jakarta_gat_multitask.pt exists= True


In [3]:
# =========================
# Load the pretrained checkpoint
# =========================
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)

hwy2id    = ckpt["hwy2id"]
id2hwy    = ckpt["id2hwy"]
token_ids = ckpt["token_ids"]
model_cfg = ckpt["model_cfg"]

# gnn.ipynb saves scalers as a flat dict of floats keyed by
# '<short>_mu' / '<short>_sd' where short in {len, wid, max, min}.
# Normalise that into a dict of ZScaler objects keyed by the long name.
from src.utils.utils import ZScaler

def _normalise_scalers(raw: dict) -> dict:
    out: dict[str, ZScaler] = {}

    # Format A: {'length': {'mu':..., 'sd':...}, ...}
    if raw and all(isinstance(v, dict) for v in raw.values()):
        for name, d in raw.items():
            s = ZScaler(); s.mu = float(d["mu"]); s.sd = float(d["sd"])
            out[name] = s
        return out

    # Format B: already ZScaler instances
    if raw and all(hasattr(v, "mu") and hasattr(v, "sd") for v in raw.values()):
        return dict(raw)

    # Format C (what gnn.ipynb saves): flat floats keyed by '<short>_mu' / '<short>_sd'.
    short2long = {"len": "length", "wid": "width", "max": "max", "min": "min"}
    buckets: dict[str, dict] = {}
    for k, v in raw.items():
        for short, long in short2long.items():
            if k == f"{short}_mu":
                buckets.setdefault(long, {})["mu"] = float(v)
            elif k == f"{short}_sd":
                buckets.setdefault(long, {})["sd"] = float(v)
    for long, d in buckets.items():
        if "mu" in d and "sd" in d:
            s = ZScaler(); s.mu = d["mu"]; s.sd = d["sd"]
            out[long] = s
    return out

scalers = _normalise_scalers(ckpt["scalers"])

for name in ("length", "width", "max", "min"):
    if name not in scalers:
        raise ValueError(f"scaler '{name}' missing in checkpoint — retrain and resave with gnn.ipynb")

print("num_highway:", len(hwy2id))
print("token_ids:  ", token_ids)
print("model_cfg:  ", model_cfg)
print("scalers:    ", {k: (float(s.mu), float(s.sd)) for k, s in scalers.items()})

num_highway: 8
token_ids:   {'HIGHWAY_MASK_ID': 7, 'LANES_MASK_ID': 3, 'LANES_MISS_ID': 4, 'ONEWAY_MASK_ID': 2, 'ONEWAY_MISS_ID': 3}
model_cfg:   {'num_highway': 8, 'hwy_emb_dim': 16, 'lanes_emb_dim': 8, 'oneway_emb_dim': 4, 'cont_dim': 12, 'hidden': 32, 'heads': 2, 'dropout': 0.1}
scalers:     {'length': (3.526740074157715, 0.936025083065033), 'width': (3.8090944290161133, 1.7127299308776855), 'max': (34.039546966552734, 24.954132080078125), 'min': (1.0850898027420044, 2.4263908863067627)}


## 1) Pick a city and derive its bounding box

In [25]:
# Change this to the city you want to evaluate on.
CITY_NAME = "Chicago, Illinois, USA"

gdf = ox.geocode_to_gdf(CITY_NAME)
polygon = gdf.geometry.iloc[0]
min_lon, min_lat, max_lon, max_lat = polygon.bounds

print(f"City:        {CITY_NAME}")
print(f"min_lat/max: {min_lat:.6f} / {max_lat:.6f}")
print(f"min_lon/max: {min_lon:.6f} / {max_lon:.6f}")

City:        Singapore
min_lat/max: 1.130361 / 1.514318
min_lon/max: 103.565515 / 104.571234


## 2) Load the graph from the Mapedia DB

In [26]:
from src.data.graph_loader import load_graph_from_db

G = load_graph_from_db(
    min_lat=min_lat, max_lat=max_lat,
    min_lon=min_lon, max_lon=max_lon,
    repo_root=REPO_ROOT,
)

Loaded graph: 287544 nodes, 381883 edges


## 3) Build the edge GeoDataFrame + merge road attributes

In [27]:
from src.data.road_attributes import build_final_gdf

DB_HOST = "cs-u-spatial-406.cs.umn.edu"
DB_NAME = "gis"
DB_USER = "gis"
DB_PASS = "gis"
DB_PORT = 5432

final_gdf = build_final_gdf(
    G,
    db_host=DB_HOST, db_name=DB_NAME,
    db_user=DB_USER, db_pass=DB_PASS,
    db_port=DB_PORT,
)
print("final_gdf rows:", len(final_gdf))

Edges: 381883, CRS: EPSG:4326
Unique osmids: 239785
Fetched road_attributes chunk 0..200000
Fetched road_attributes chunk 200000..239785
road_attributes rows: 182157, unique osm_id: 182157

Missing values:
highway      152253
lanes        221689
width        381001
oneway       266294
length            0
max_speed    273300
min_speed    381883
dtype: int64
final_gdf rows: 381883


## 4) Prepare the `Data` object using the FROZEN NYC vocab + scalers

`prepare_cross_city_data` does NOT refit anything: unknown highway types fall back to `__UNK__`, and continuous features are normalised with NYC's `mu`/`sd`.

In [28]:
from src.training.dataset import prepare_cross_city_data

data_city = prepare_cross_city_data(
    final_gdf,
    hwy2id=hwy2id,
    scalers=scalers,
    device=device,
)
print(f"{CITY_NAME}: {data_city.num_nodes} nodes, {data_city.edge_index.shape[1]} edges")

# How many edges had highway types the model has never seen before?
unk_id = hwy2id["__UNK__"]
n_unk = int((data_city.highway_in == unk_id).sum().item())
print(f"UNK highway edges: {n_unk} ({100.0 * n_unk / data_city.num_nodes:.2f}%)")

Singapore: 381883 nodes, 1507020 edges
UNK highway edges: 152253 (39.87%)


## 5) Rebuild the model and load the pretrained weights

In [29]:
from src.models.multi_attr_gat import MultiAttrGAT

model = MultiAttrGAT(
    num_highway=model_cfg["num_highway"],
    hwy_emb_dim=model_cfg["hwy_emb_dim"],
    lanes_emb_dim=model_cfg["lanes_emb_dim"],
    oneway_emb_dim=model_cfg["oneway_emb_dim"],
    cont_dim=model_cfg["cont_dim"],
    hidden=model_cfg["hidden"],
    heads=model_cfg["heads"],
    dropout=model_cfg["dropout"],
).to(device)

missing, unexpected = model.load_state_dict(ckpt["model_state"], strict=False)
print("missing keys:   ", missing)
print("unexpected keys:", unexpected)
model.eval()

missing keys:    []
unexpected keys: []


MultiAttrGAT(
  (hwy_emb): Embedding(8, 16)
  (lanes_emb): Embedding(5, 8)
  (oneway_emb): Embedding(4, 4)
  (gat1): GATv2Conv(40, 32, heads=2)
  (gat2): GATv2Conv(64, 32, heads=2)
  (head_highway): Linear(in_features=32, out_features=8, bias=True)
  (head_lanes): Linear(in_features=32, out_features=3, bias=True)
  (head_oneway): Linear(in_features=32, out_features=1, bias=True)
  (head_width): Linear(in_features=32, out_features=1, bias=True)
  (head_max): Linear(in_features=32, out_features=1, bias=True)
  (head_min): Linear(in_features=32, out_features=1, bias=True)
)

## 6) Evaluate with fixed masks (same metrics as `gnn.ipynb`)

In [30]:
from src.training.masking import make_fixed_masks
from src.training.losses import evaluate_with_masks

P_MASK    = ckpt.get("meta", {}).get("p_mask", 0.30)
TEST_SEED = 2025                                         # same seed as gnn.ipynb
num_highway = model_cfg["num_highway"]

masks = make_fixed_masks(data_city, p_mask=P_MASK, seed=TEST_SEED)

total, losses, metrics = evaluate_with_masks(
    model,
    data_city,
    masks,
    num_highway_classes=num_highway,
    device=device,
    highway_mask_id=token_ids["HIGHWAY_MASK_ID"],
    lanes_mask_id=token_ids["LANES_MASK_ID"],
    oneway_mask_id=token_ids["ONEWAY_MASK_ID"],
)

print(f"\n=== {CITY_NAME} — fixed-mask evaluation (p_mask={P_MASK}, seed={TEST_SEED}) ===")
print(f"total loss: {total:.4f}")
print("per-task losses:")
for k, v in losses.items():
    print(f"  {k}: {v:.4f}")
print("metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}" if not np.isnan(v) else f"  {k}: NaN")


=== Singapore — fixed-mask evaluation (p_mask=0.3, seed=2025) ===
total loss: 37.2175
per-task losses:
  hwy: 4.6631
  lan: 0.8201
  onw: 0.2005
  wid: 0.6647
  max: 1.8008
  min: 0.0000
metrics:
  hwy_macro_f1: 0.1203
  lan_macro_f1: 0.7249
  onw_auroc: 0.9276
  wid_mae_m: 1.0640
  max_mae: 2.2208
  min_mae: NaN


## 7) (optional) Evaluate on a batch of cities

Pulls each city's bbox from OSMnx, fetches the graph, prepares the data with the frozen vocab/scalers, and runs the same evaluation.

In [ ]:
# CITIES = [
#     "Cairo, Egypt",
#     "Manhattan, New York City, USA",
#     "Bangkok, Thailand",
# ]

# rows = []
# for name in CITIES:
#     try:
#         g = ox.geocode_to_gdf(name)
#         minx, miny, maxx, maxy = g.geometry.iloc[0].bounds

#         G_c = load_graph_from_db(
#             min_lat=miny, max_lat=maxy,
#             min_lon=minx, max_lon=maxx,
#             repo_root=REPO_ROOT,
#         )
#         fgdf = build_final_gdf(
#             G_c,
#             db_host=DB_HOST, db_name=DB_NAME,
#             db_user=DB_USER, db_pass=DB_PASS, db_port=DB_PORT,
#         )
#         data_c = prepare_cross_city_data(fgdf, hwy2id=hwy2id, scalers=scalers, device=device)
#         m_c = make_fixed_masks(data_c, p_mask=P_MASK, seed=TEST_SEED)
#         _, _, met = evaluate_with_masks(
#             model, data_c, m_c,
#             num_highway_classes=num_highway, device=device,
#             highway_mask_id=token_ids["HIGHWAY_MASK_ID"],
#             lanes_mask_id=token_ids["LANES_MASK_ID"],
#             oneway_mask_id=token_ids["ONEWAY_MASK_ID"],
#         )
#         met["city"] = name
#         met["n_edges"] = int(data_c.num_nodes)
#         rows.append(met)
#         print(f"[done] {name}: {met}")
#     except Exception as e:
#         print(f"[fail] {name}: {e}")

# results = pd.DataFrame(rows).set_index("city")[
#     ["n_edges", "hwy_macro_f1", "lan_macro_f1", "onw_auroc", "wid_mae_m", "max_mae", "min_mae"]
# ]
# results